# 01 -- FrankenStat (composite) strategy on simulated data

The composite combination strategy treats every PTA's timing model as untouchable: each parameter from each PTA is preserved, and parameters that happen to share a name across PTAs are kept as separate, PTA-suffixed entries. The result is *not* an astrophysically self-consistent timing model -- it is a Frankenstein assembly with one fitted offset per PTA per parameter. The community calls this the **Borg method**, **FrankenStat**, or simply the **composite** approach. The paper describing the method can be found at: https://arxiv.org/abs/2512.14807

Why use it?

* **Robustness.** Because every PTA keeps its own model, there are no model-consistency requirements to run the method.
* **Simplicity.** Fewer moving parts means fewer components that can break.
* **Diagnostic value.** Inspecting the spread of, say, `PMRA_epta_dr2` vs `PMRA_nanograv_9y` can hold valuable information when diagnosing issues.

## What this notebook does

Rather than working on the real IPTA-DR2 release, this notebook drives the **FrankenStat simulation pipeline** vendored under [`./frankenstat/`](frankenstat/). That code is a snapshot of David Wright's [`frankenstat-paper-1`](https://github.com/davecwright3/frankenstat-paper-1) repo (see [`frankenstat/README.md`](frankenstat/README.md)). The notebook walks through:

1. Simulate a small (`npsr=10`) PTA from priors with an injected GW background.
2. Split the simulated PTA into three sub-PTAs that all observe the same pulsars.
3. Frankenize the three sub-PTAs into composite **FrankenPulsars**.
4. Inspect a single FrankenPulsar -- its TOA stack, its block-diagonal design matrix, and its merged noise dictionary.

The full FrankenStat paper pipeline continues with single-pulsar noise analysis, an HD max-likelihood step, and an optimal-statistic p-value -- those are out of scope for this hour-long session. Dave plans to extend the notebook with those stages later; the relevant entry points already live in [`frankenstat/analysis.py`](frankenstat/analysis.py) and [`frankenstat/discovery_os_gx2.py`](https://github.com/davecwright3/frankenstat-paper-1/blob/master/discovery_os_gx2.py).

**Independence note.** This notebook does not depend on `00_setup.ipynb` or on the `data/ipta-dr2` submodule -- everything is generated locally. Notebooks 02 and 03 (the MetaPulsar walkthrough on real IPTA-DR2) are the ones that need the submodule.

## Step 1 -- Imports and config

We add `./frankenstat/` to `sys.path` so the vendored modules can use bare sibling imports the way Dave's upstream layout does. The vendored `simulation.py` and `analysis.py` carry tiny try/except shims so they import cleanly even if `cyclopts` or `schwimmbad` are missing -- see [`frankenstat/_shims.py`](frankenstat/_shims.py).

In [ ]:
import sys
import warnings
from pathlib import Path

import loguru
import numpy as np

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

FRANKEN_DIR = Path("frankenstat").resolve()
if str(FRANKEN_DIR) not in sys.path:
    sys.path.insert(0, str(FRANKEN_DIR))


def quiet_loguru(level: str = "WARNING") -> None:
    loguru.logger.remove()
    loguru.logger.add(sys.stdout, level=level)


quiet_loguru()

import simulation as sim
import discovery_utils as du

DATA_DIR = FRANKEN_DIR / "frankenstat_demo_outputs" / "realization-0"
NPSR = 10
GW_LOG10_A = -15.7
GW_GAMMA = 13.0 / 3.0
RNG_SEED = 42

print(f"FRANKEN_DIR = {FRANKEN_DIR}")
print(f"DATA_DIR    = {DATA_DIR}")
print(f"npsr={NPSR}, gw_log10_A={GW_LOG10_A}, gw_gamma={GW_GAMMA:.4f}, rng_seed={RNG_SEED}")

## Step 2 -- Simulate a small PTA from priors

[`simulation.simulate_pull_from_priors_parallel`](frankenstat/simulation.py) generates a Fibonacci-lattice of `npsr` pulsars with realistic spin/timing parameters, draws white-noise (EFAC/EQUAD) and intrinsic red-noise parameters, then injects an HD-correlated GW background of amplitude `gw_log10_A` and spectral index `gw_gamma`. The output is a directory layout that the rest of the pipeline expects:

```
frankenstat_demo_outputs/realization-0/
    fake_pars/        # fitted .par files
    fake_tims/        # fitted .tim files
    fake_feathers/    # enterprise FeatherPulsar.feather files (one per pulsar)
    injected-parameters.json
    ...
```

We disable MPI (`use_mpi=False`) so the vendored `_shims.MultiPool` (a serial `map`) is used. At `npsr=10` this takes ~1-2 min on a laptop.

If the simulation has already run, we skip it -- delete `frankenstat/frankenstat_demo_outputs/` to force a regeneration.

In [ ]:
if (DATA_DIR / "fake_feathers").exists() and any(
    (DATA_DIR / "fake_feathers").glob("*.feather")
):
    print(f"Reusing existing simulation at {DATA_DIR}")
else:
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    sim.simulate_pull_from_priors_parallel(
        data_dir=str(DATA_DIR),
        npsr=NPSR,
        gw_log10_A=GW_LOG10_A,
        gw_gamma=GW_GAMMA,
        ecorr_include=False,
        wn_const=True,
        rn_include=True,
        rn_comp=30,
        rng_seed=RNG_SEED,
        use_mpi=False,
    )

feather_files = sorted((DATA_DIR / "fake_feathers").glob("*.feather"))
print(f"\nSimulated {len(feather_files)} pulsars in {DATA_DIR / 'fake_feathers'}")
for f in feather_files[:5]:
    print(f"  {f.name}")
if len(feather_files) > 5:
    print(f"  ... and {len(feather_files) - 5} more")

## Step 3 -- Split the combined PTA into three sub-PTAs

[`simulation.split_simulated_pta_parallel`](frankenstat/simulation.py) carves the combined dataset into three equal-time sub-PTAs (`pta_1`, `pta_2`, `pta_3`) that all observe the same pulsars. This mimics the real-world situation where EPTA, NANOGrav, and PPTA each see (mostly) the same set of millisecond pulsars but with different cadence and observatory backends.

The result is the per-PTA feather directories that the FrankenStat combination consumes.

In [ ]:
if all(
    (DATA_DIR / f"fake_feathers_pta_{i}").exists()
    and any((DATA_DIR / f"fake_feathers_pta_{i}").glob("*.feather"))
    for i in (1, 2, 3)
):
    print("Reusing existing per-PTA splits")
else:
    sim.split_simulated_pta_parallel(data_dir=str(DATA_DIR), use_mpi=False)

for i in (1, 2, 3):
    n = len(list((DATA_DIR / f"fake_feathers_pta_{i}").glob("*.feather")))
    print(f"  pta_{i}: {n} pulsars")

## Step 4 -- Frankenize

[`simulation.frankenize_split_pta`](frankenstat/simulation.py) walks the three sub-PTAs and, for every pulsar that appears in more than one of them, calls [`frankenstat.frankenize_duplicate_pulsar`](frankenstat/frankenstat.py) to glue the per-PTA observations into a single composite pulsar. The mechanics:

* TOAs, residuals, and TOA uncertainties are simply **concatenated** across the three PTAs.
* The design matrix is assembled as a **block-diagonal** of the per-PTA design matrices, so each PTA's timing-model parameters only see their own TOAs.
* Pulsar position / DM / parallax values are **truncated to the first decimal place where the three PTAs disagree** (`truncate_vals` in `frankenstat.py`) so we don't pretend to higher precision than the inputs support.
* Per-PTA noise dictionaries are merged with PTA-suffix stripping (`_epta`/`_ppta`/`_ng`).

The output is `franken_psrs/franken-J<name>.feather` for every pulsar that had more than one PTA observing it, plus single-PTA pulsars copied through unchanged.

In [ ]:
if (DATA_DIR / "franken_psrs").exists() and any(
    (DATA_DIR / "franken_psrs").glob("franken-*.feather")
):
    print("Reusing existing FrankenPulsars")
else:
    sim.frankenize_split_pta(data_dir=DATA_DIR)

franken_files = sorted((DATA_DIR / "franken_psrs").glob("franken-*.feather"))
print(f"Created {len(franken_files)} FrankenPulsars")
for f in franken_files[:5]:
    print(f"  {f.name}")
if len(franken_files) > 5:
    print(f"  ... and {len(franken_files) - 5} more")

## Step 5 -- Inspect a FrankenPulsar

Pick the first FrankenPulsar and look at the structures the combination produced. The interesting things to notice:

* `toas.shape` is the **sum** of the three sub-PTAs' TOA counts.
* `Mmat.shape` is `(N_TOA, 3 * N_fit_per_PTA)` -- block diagonal, *not* shared.
* `noisedict` carries one `efac`/`equad` *per sub-PTA*, plus one shared red-noise pair (`red_noise_log10_A`, `red_noise_gamma`).
* `backend_flags` tells you which TOA came from which sub-PTA (`fake_1_AXIS`, `fake_2_AXIS`, `fake_3_AXIS`).

In [ ]:
franken_psrs = du.read_pulsar_feathers(DATA_DIR / "franken_psrs", prefix="franken")
psr = franken_psrs[0]

print(f"FrankenPulsar    : {psr.name}")
print(f"  TOA count      : {psr.toas.size}")
print(f"  Mmat shape     : {psr.Mmat.shape}  (block-diagonal across 3 sub-PTAs)")
print(f"  Span           : {(psr.toas.max() - psr.toas.min()) / 86400 / 365.25:.2f} yr")
print()

unique, counts = np.unique(psr.backend_flags, return_counts=True)
print("  TOAs per sub-PTA backend flag:")
for b, c in zip(unique, counts):
    print(f"    {b:<20s} {c:>5d}")
print()

print("  noisedict (selected entries):")
for k in list(psr.noisedict)[:8]:
    print(f"    {k:<55s} = {psr.noisedict[k]:+.4f}")

## Step 6 -- Visualize the FrankenPulsar

Two quick plots make the structure concrete:

1. **Residuals coloured by source sub-PTA** -- the same pulsar seen by three independent backends.
2. **Design-matrix sparsity (`spy(Mmat)`)** -- the block-diagonal pattern is the visual signature of FrankenStat: each sub-PTA's columns are non-zero only over its own rows.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax_res, ax_mmat) = plt.subplots(1, 2, figsize=(11, 4.5))

mjd = psr.toas / 86400.0
for backend in np.unique(psr.backend_flags):
    mask = psr.backend_flags == backend
    ax_res.errorbar(
        mjd[mask],
        psr.residuals[mask] * 1e6,
        yerr=psr.toaerrs[mask] * 1e6,
        fmt=".",
        ms=3,
        alpha=0.6,
        label=backend,
    )
ax_res.set_xlabel("MJD")
ax_res.set_ylabel(r"Residual [$\mu$s]")
ax_res.set_title(f"{psr.name}: residuals per source sub-PTA")
ax_res.legend(loc="best", fontsize=9)
ax_res.grid(alpha=0.3)

ax_mmat.spy(psr.Mmat != 0, aspect="auto", markersize=1)
ax_mmat.set_xlabel("Timing-model column")
ax_mmat.set_ylabel("TOA index")
ax_mmat.set_title(f"{psr.name}: design matrix (block-diagonal)")

plt.tight_layout()

## What's next

From here the [`frankenstat/analysis.py`](frankenstat/analysis.py) continues with:

1. **Single-pulsar noise analysis (SPNA)** on the combined PTA, on each sub-PTA, and on the FrankenPulsars.
2. **HD max-likelihood** estimation of common-red-noise (CRN) parameters with `analysis.run_array_max_like`.
3. **Optimal-statistic p-value** comparison across the five PTAs (combined / Franken / pta_1 / pta_2 / pta_3) with `discovery_os_gx2.get_pvalues_five_pta`.

Those steps require GPU acceleration so we skip these for now. Instead, let's see if we can do the same as above with MetaPulsar.

Notebooks [`02_metapulsar_consistent.ipynb`](02_metapulsar_consistent.ipynb) and [`03_consistency_checks.ipynb`](03_consistency_checks.ipynb) switch gears entirely -- they leave simulation behind and walk through MetaPulsar's *consistent* combination strategy on the real IPTA-DR2 release.

## Step 7 -- Round-trip with MetaPulsar's `composite` strategy

The FrankenStat pipeline above is a self-contained recipe: simulate -> split -> frankenize. MetaPulsar implements *the same combination strategy* under the name `combination_strategy="composite"`. That gives us a clean cross-check: if we point MetaPulsar at the per-sub-PTA `par`/`tim` files we just generated, do we get back something equivalent to the FrankenPulsar that the simulation pipeline produced?

> **Automated Discovery** MetaPulsar has automated file discovery. More on that in the MetaPulsar tutorial. Here we just manually set a regexp consistent with the FrankenStat simulation pipeline, and use a MetaPulsar routine
> The uses lower-level [`discover_files`](../../src/metapulsar/file_discovery_service.py) functionality for our convenience

In [ ]:
from metapulsar import (
    create_metapulsar,
    discover_files,
    filter_file_data_by_pulsars,
)

quiet_loguru("WARNING")

# This is manually set here for the purposes of this notebook. Usually you'd use MetaPulsar's automated discovery for an existing PTA dataset.
# NOTE: the fact that this is necessary is probably a testament that the MetaPulsar routines could be made more general to accommodate certain simulation pipeline layouts.
sim_layout = {
    f"sim_pta_{i}": {
        "base_dir": str(DATA_DIR),
        "par_pattern": rf"fake_pars_pta_{i}/([BJ]\d{{4}}[+-]\d{{2,4}})\.par",
        "tim_pattern": rf"fake_tims_pta_{i}/([BJ]\d{{4}}[+-]\d{{2,4}})\.tim",
        "timing_package": "pint",
        "description": f"Simulated sub-PTA {i}",
    }
    for i in (1, 2, 3)
}

sim_file_data = discover_files(sim_layout, verbose=True)

## Step 8 -- Build a composite MetaPulsar for one pulsar

We pick the same target pulsar we inspected in Step 5 and ask MetaPulsar to combine its three per-sub-PTA timing models with `combination_strategy="composite"`. By design, that mirrors what `frankenstat.frankenize_duplicate_pulsar` did: every parameter from every sub-PTA is preserved as a separately-suffixed entry, and the design matrix becomes block-diagonal across the three sources.

In [ ]:
TARGET = psr.name

quiet_loguru("ERROR")
target_file_data = filter_file_data_by_pulsars(sim_file_data, [TARGET])
mp = create_metapulsar(
    file_data=target_file_data,
    combination_strategy="composite",
)
quiet_loguru("WARNING")

print(f"MetaPulsar (composite): {mp.name}")
print(f"  TOA count        : {mp.toas.size}")
print(f"  Mmat shape       : {mp.Mmat.shape}")
print(f"  fitpars (n={len(mp.fitpars)}):")
for fp_name in mp.fitpars:
    print(f"    {fp_name}")

## Step 9 -- Compare against the FrankenPulsar

Both objects describe the same pulsar built from the same three sets of par/tim files with the same combination strategy, so first-order quantities (TOA count, Mmat shape, number of fit parameters) should agree exactly. We expect *cosmetic* differences in row/column ordering and parameter naming, plus tiny numerical differences in residuals from independent enterprise/PINT instantiations. We surface those without trying to fix them.

In [ ]:

fp = next(p for p in franken_psrs if p.name == TARGET)

print("=== Shape and ordering ===")
print(f"  TOA count          MP={mp.toas.size:>5d}    FP={fp.toas.size:>5d}    "
      f"match={mp.toas.size == fp.toas.size}")
print(f"  Mmat shape         MP={str(mp.Mmat.shape):>10s}  FP={str(fp.Mmat.shape):>10s}  "
      f"match={mp.Mmat.shape == fp.Mmat.shape}")
print(f"  TOAs time-sorted   MP={bool(np.all(np.diff(mp.toas) >= 0))}    "
      f"FP={bool(np.all(np.diff(fp.toas) >= 0))}")

# FrankdenStat and MetaPulsar use differen TOA ordering. So we create a mapping between them
# Since FrankenStat orders by PTA, we keep that, and just reorder the MetaPulsar TOAs to match
fp_order = np.argsort(fp.toas)
mp_order = np.empty_like(fp_order)
mp_order[fp_order] = np.arange(fp_order.size)
fp_order = np.arange(fp_order.size)

toa_match = np.allclose(mp.toas[mp_order], fp.toas[fp_order])
print(f"  Sorted TOAs equal  {toa_match}")

resid_diff = np.max(np.abs(mp.residuals[mp_order] - fp.residuals[fp_order]))
resid_scale = np.max(np.abs(fp.residuals))
print(f"  max |Dres|         {resid_diff:.3e} s   (residual scale {resid_scale:.3e} s, "
      f"relative {resid_diff / resid_scale:.1e})")

print()
print("=== Per-source TOA partition ===")
print(f"  unique backend_flags (MP): {sorted(np.unique(mp.backend_flags).tolist())}")
print(f"  unique backend_flags (FP): {sorted(np.unique(fp.backend_flags).tolist())}")
for flag in sorted(np.unique(fp.backend_flags)):
    n_mp = int(np.sum(mp.backend_flags == flag))
    n_fp = int(np.sum(fp.backend_flags == flag))
    print(f"    {flag:<14s} MP={n_mp:>4d}  FP={n_fp:>4d}  match={n_mp == n_fp}")

print()
print("=== Fit-parameter naming ===")
print("  MP.fitpars (flat, suffixed):")
print(f"    {mp.fitpars}")
print("  FP.fitpars (list of per-sub-PTA lists, no suffix):")
print(f"    {fp.fitpars}")

### Mmat side-by-side

The block-diagonal-ness of the design matrix is the visual signature of the composite strategy. We `spy` both Mmats with rows sorted by TOA time, which lets us compare the two block structures directly even though MetaPulsar and FrankenStat lay the columns out in different orders.

In [ ]:
fig, (ax_mp, ax_fp) = plt.subplots(1, 2, figsize=(11, 5), sharey=True)

ax_mp.spy(mp.Mmat[mp_order] != 0, aspect="auto", markersize=1)
ax_mp.set_title(f"MetaPulsar (composite)  shape={mp.Mmat.shape}")
ax_mp.set_xlabel(f"fit parameter index ({len(mp.fitpars)} total)")
ax_mp.set_ylabel("TOA index (time-sorted)")

ax_fp.spy(fp.Mmat != 0, aspect="auto", markersize=1)
ax_fp.set_title(f"FrankenPulsar          shape={fp.Mmat.shape}")
ax_fp.set_xlabel(f"fit parameter index ({sum(len(x) for x in fp.fitpars)} total)")

fig.suptitle(f"Mmat block structure for {TARGET}  (rows sorted by TOA time)")
fig.tight_layout()

### What we observed

- **TOA set** is identical between the two pipelines once both arrays are time-sorted. The native ordering differs: MetaPulsar time-sorts internally, while the FrankenPulsar keeps the per-sub-PTA blocks in concatenation order.
- **Design matrix** has the same shape and the same block-diagonal structure. Column ordering and labelling differ -- MetaPulsar exposes a flat `fitpars` list with `_sim_pta_N` suffixes (including a per-block `Offset_sim_pta_N`), while the FrankenPulsar keeps `fitpars` as a list of three per-sub-PTA lists with the original parameter names. Both encode the same 18 columns.
- **Residuals** agree to within 1ns
- **Per-sub-PTA TOA partition** (counted from `backend_flags`) matches exactly.

The two approaches yield fully consistent results

In [ ]:
# Residuals are the same to within 1ns:
np.abs((mp.residuals[mp_order] - fp.residuals[fp_order]))[:10], np.abs((mp.residuals[mp_order] - fp.residuals[fp_order])).max()